In [2]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, length, count

TITULOS_RUIDO = ["Anuncios Google", "Open job preview"]

def corregir_outliers(df):
    antes = df.count()

    # 1. Eliminar títulos de ruido conocido
    for ruido in TITULOS_RUIDO:
        df = df.filter(~col("titulo").contains(ruido))

    # 2. Calcular largo del título
    df = df.withColumn("largo_titulo", length(col("titulo")))

    # 3. IQR sobre largo_titulo para detectar outliers extremos
    cuantiles = df.approxQuantile("largo_titulo", [0.25, 0.75], 0.01)
    Q1, Q3 = cuantiles[0], cuantiles[1]
    IQR = Q3 - Q1
    limite_superior = Q3 + 3 * IQR

    print(f"Q1={Q1} | Q3={Q3} | IQR={IQR} | Limite superior: {limite_superior}")

    df = df.filter(col("largo_titulo") <= limite_superior)
    df = df.drop("largo_titulo")

    despues = df.count()
    print(f"Antes : {antes}")
    print(f"Despues: {despues}")
    print(f"Outliers eliminados: {antes - despues}")

    return df